# 13. Gamma Framework — Remediation Report (v0.2)

**Purpose**: rectify the anomalies identified in NB 12 *and* the parametrization mistake found in v0.1 of this report (variable L instead of fixed capital). Final verdict supersedes both NB 12 and v0.1.

**Caveat upfront**: only USDC/WETH 0.05% (`0x88e6a0c2…`) is indexed. The 0.3% pool (`0x8ad599c…`) returns 0 rows from `mart_price_intraday`; cross-fee-tier generalization is out of scope until that data lands.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault('CLICKHOUSE_HOST', 'localhost')
os.environ.setdefault('CLICKHOUSE_PORT', '8124')
os.environ.setdefault('CLICKHOUSE_DATABASE', 'analytics')
os.environ.setdefault('CLICKHOUSE_USER', 'default')
os.environ.setdefault('CLICKHOUSE_PASSWORD', '1234')

POOL = '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640'
START_DATE = '2025-11-05'
END_DATE   = '2026-05-03'
CAPITAL = 1_000_000

## §1 — What v0.1 got wrong

v0.1 of this report fixed the v3 capital math (token0/token1 swap due to `close_price = USDC/WETH`) but kept a hidden flaw: **L_position = L_pool_active for every range**. Consequences:

1. Capital varied 25× across ranges ($8M at ±3% → $200M at fitted-wide).
2. ±20% and fitted-wide reported **identical PnL** (because TIR=100% in both → same fees and gc on the same L).
3. The optimization-of-range question was never answered: the table compared bets of different sizes.

v0.2 fixes this by **fixing capital at $1M** and back-solving L from the v3 amount equations:

```
capital = L · [(sp − √Pa) + (√Pb − sp)/(sp · √Pb) · P0]
L = capital / coef
```

where `sp = √close_price`, `Pa, Pb` in close_price (USDC/WETH) units, and `P0` is the prior-day close used as the rebalance point.

## §2 — Capital-constant sweep ($1M, daily rebal, 153 days)

Replays `python/scripts/concentrated_position_remediation_v2.py`. Position fees apportioned as `pool_fees · (L_position / L_pool_active) · TIR`.

In [ ]:
sweep = pd.DataFrame([
    {'range': '±3%',    'TIR_pct': 80.7, 'days_oor': 74, 'lev': 67.15, 'L':663889.5, 'share_pct':33.13, 'fees_mean': 7978, 'gc_mean': 7798, 'FGY': 1.023, 'FGY_diff': 1.247, 'PnL': 27_472},
    {'range': '±5%',    'TIR_pct': 92.0, 'days_oor': 37, 'lev': 40.47, 'L':400115.6, 'share_pct':19.97, 'fees_mean': 5840, 'gc_mean': 5449, 'FGY': 1.072, 'FGY_diff': 1.304, 'PnL': 59_858},
    {'range': '±10%',   'TIR_pct': 99.5, 'days_oor':  2, 'lev': 20.44, 'L':202075.4, 'share_pct':10.08, 'fees_mean': 3490, 'gc_mean': 3028, 'FGY': 1.153, 'FGY_diff': 1.398, 'PnL': 70_671},
    {'range': '±20%',   'TIR_pct':100.0, 'days_oor':  0, 'lev': 10.38, 'L':102614.0, 'share_pct': 5.12, 'fees_mean': 1870, 'gc_mean': 1546, 'FGY': 1.209, 'FGY_diff': 1.466, 'PnL': 49_499},
    {'range': 'wide',   'TIR_pct':100.0, 'days_oor':  0, 'lev':  2.83, 'L': 27977.1, 'share_pct': 1.40, 'fees_mean':  509, 'gc_mean':  421, 'FGY': 1.209, 'FGY_diff': 1.466, 'PnL': 13_480},
])
sweep['APR_pct'] = sweep['PnL'] / CAPITAL * (365 / 153) * 100
sweep

### Read-out

1. **Every range is profitable at $1M capital.** The earlier «structurally to perte» framing was driven by paramétrisation, not by the underlying economics.
2. **Optimum is ±10% daily-rebal**: $71k / $1M / 153d ≈ **16.9% APR**. TIR=99.5% (only 2 OOR days), leverage=20×.
3. **±3% loses to ±10%** despite 3× higher leverage — TIR drops to 80.7%, fees forfeited dominate.
4. **±20% and fitted-wide share FGY=1.209**: TIR=100% in both, same fees-per-L and gc-per-L. Absolute PnL differs only because L (and hence skin in the game) differs 3.7×.
5. **FGY_diffusion ranges 1.25–1.47** across all widths. Diffusion is comfortably profitable; jumps absorb only 16-17% of total gc, leaving total FGY ≥ 1.

## §3 — Jump-rule audit (replaces NB 12 §6)

v0.1 claimed `jump_intensity_lag1 > p75_30d` saved $765k. That claim is now retracted — it was an artifact of the v0.1 negative-PnL regime where any random skip reduces losses.

Three audit checks:

### §3.a Lookahead

`p75_30d` computed via `shift(1).rolling(30, min_periods=10).quantile(0.75)` — strictly past-only. Decision for day t uses observations from [t−30, t−1] only. **No data leakage.**

### §3.b Jump-intensity autocorrelation

In [ ]:
acf = pd.DataFrame({
    'lag': [1, 3, 7],
    'autocorrelation': [+0.0389, -0.0514, +0.0635],
    'noise_band_2σ_N153': [2/np.sqrt(153)] * 3,
})
acf['significant_at_2σ'] = acf['autocorrelation'].abs() > acf['noise_band_2σ_N153']
acf

**Conclusion**: jump intensity is approximately white noise at all three lags. The premise of jump clustering — central to the rule — is **not supported by the data**. Without persistence, knowing today's jump tells you nothing about tomorrow's.

### §3.c Threshold sensitivity (reference run = ±10%)

In [ ]:
sens = pd.DataFrame([
    {'q': 0.50, 'n_skipped': 74, 'PnL_baseline': 70_671, 'PnL_with_rule': 740,    'savings': -69_931},
    {'q': 0.60, 'n_skipped': 61, 'PnL_baseline': 70_671, 'PnL_with_rule': 38_353, 'savings': -32_318},
    {'q': 0.70, 'n_skipped': 41, 'PnL_baseline': 70_671, 'PnL_with_rule': 49_535, 'savings': -21_136},
    {'q': 0.75, 'n_skipped': 40, 'PnL_baseline': 70_671, 'PnL_with_rule': 45_607, 'savings': -25_064},
    {'q': 0.80, 'n_skipped': 31, 'PnL_baseline': 70_671, 'PnL_with_rule': 38_993, 'savings': -31_678},
    {'q': 0.90, 'n_skipped': 18, 'PnL_baseline': 70_671, 'PnL_with_rule': 58_784, 'savings': -11_887},
])
sens

**At every threshold the rule destroys value** (savings < 0 throughout). This is the expected outcome given the autocorrelation result: there is no statistical hook for the rule to grab onto.

Why was v0.1 «savings ≈ +$765k» wrong?  When baseline PnL is structurally negative (v0.1 had Σ_pnl ≈ −$1.7M), any rule that randomly drops days mechanically improves the figure — it stops accumulating losses on the skipped days. v0.1 mistook this arithmetic effect for predictive value. With a profitable baseline (v0.2: +$70k), the same rule has nothing to skip *toward* and instead drops winners.

## §4 — HAR-RV rule (unchanged from prior audit)

HAR rule loses ~$1.5M against always-LP on the v0.1 (negative-PnL) baseline. With v0.2's positive baseline, HAR almost certainly loses even more in absolute terms — but the *mechanism* is the same finding regardless of baseline sign:

- HAR captures vol *persistence* (today's RV high → tomorrow's HAR forecast high).
- High-RV days correlate with **high fee income**, not with **jumps** (which are by construction the unforecastable component of RV).
- The rule therefore skips high-fee days while doing nothing about jump days. 

Combined with the autocorrelation finding above (jumps don't persist), there is **no signal in our window** that delivers a profitable skip rule.

## §5 — Final synthesis table

In [ ]:
# Combined: baseline PnL vs jump_rule(p75) PnL across ranges, capital-constant
synth = pd.DataFrame([
    {'pool':'USDC/WETH 0.05%','range':'±3%',   'lev': 67.2, 'TIR_pct': 80.7, 'FGY':1.023, 'FGY_diff':1.247, 'PnL_base': 27_472, 'PnL_jumprule_p75': 14_746, 'savings': -12_726},
    {'pool':'USDC/WETH 0.05%','range':'±5%',   'lev': 40.5, 'TIR_pct': 92.0, 'FGY':1.072, 'FGY_diff':1.304, 'PnL_base': 59_858, 'PnL_jumprule_p75': 54_471, 'savings':  -5_388},
    {'pool':'USDC/WETH 0.05%','range':'±10%',  'lev': 20.4, 'TIR_pct': 99.5, 'FGY':1.153, 'FGY_diff':1.398, 'PnL_base': 70_671, 'PnL_jumprule_p75': 45_607, 'savings': -25_064},
    {'pool':'USDC/WETH 0.05%','range':'±20%',  'lev': 10.4, 'TIR_pct':100.0, 'FGY':1.209, 'FGY_diff':1.466, 'PnL_base': 49_499, 'PnL_jumprule_p75': 35_332, 'savings': -14_167},
    {'pool':'USDC/WETH 0.05%','range':'wide',  'lev':  2.8, 'TIR_pct':100.0, 'FGY':1.209, 'FGY_diff':1.466, 'PnL_base': 13_480, 'PnL_jumprule_p75':  9_632, 'savings':  -3_848},
])
synth['APR_base_pct']  = synth['PnL_base']         / CAPITAL * (365/153) * 100
synth['APR_rule_pct']  = synth['PnL_jumprule_p75'] / CAPITAL * (365/153) * 100
synth

## §6 — Revised verdict

| Question | NB 12 | v0.1 | **v0.2 (final)** |
|---|---|---|---|
| Diffusion-only profitable? | No (FGYd=0.81) | Yes barely (FGYd≈1.04) | **Yes solidly** (FGYd 1.25–1.47 across widths) |
| Total LP profitable on this pool/window? | No (FGY=0.81) | No (≈ -$1.5M) | **Yes** ($13k–$71k / $1M / 153d, peaks at ±10%) |
| Jump share of gc? | 16.7% | 16-17% | 16-17% (unchanged) |
| HAR-RV signal? | Loses $1.5M | Loses $1.5M | Loses, mechanism understood |
| jump_intensity_lag1 > p75 signal? | n/a | «Saves $765k» (artefact) | **Loses at every threshold** |
| Optimum range? | n/a | n/a | **±10% daily-rebal** |

**Bottom line for the POC**: on USDC/WETH 0.05% over Nov 2025 → May 2026, a daily-rebalanced concentrated LP at $1M is profitable across all reasonable widths, with a clean optimum near ±10% (≈17% APR). Jumps account for ~17% of gamma cost but do not cluster autocorrelationally, so neither vol-persistence (HAR) nor jump-persistence rules can be exploited at this sample size. The structural-loss claim of NB 12 is retracted.

**What the POC does NOT demonstrate**:
- Generalization to ETH/USDC 0.3% (data unavailable).
- Robustness to rebalancing cost (gas + swap fees not modelled — at ±10% daily, ~$200/day overhead would compress APR by ~7pp).
- Out-of-sample on a different volatility regime.
- Sensitivity of the diffusion/jump split to the BV π/2 normalization.

## §7 — Caveats explicit
1. **Single pool / single window** (153 days, USDC/WETH 0.05%).
2. **Fees apportioned via `pool_fees × share × TIR`**: assumes the position's L is a small enough fraction of the active pool that it doesn't move the price impact of swaps. At share ≈ 33% (±3%) this approximation is rougher.
3. **No rebalancing cost modelled**. Daily rebal at ±10% needs ~1 swap/day; at typical sizes that's $50–200/day — non-negligible against $200/day fees mean.
4. **BV decomposition** uses standard π/2 normalization. Robustness to alt-normalization (median-RV, MinRV) untested.
5. **Autocorrelation N=153 has 2σ noise band ≈ ±0.16**. None of the lags clear that band; the claim is «consistent with white noise», not «proven white noise».